# 🔍 VerifAI — Complete Training Notebook
### Multimodal Misinformation Detection: CLIP + GNN + XAI

**Run cells top to bottom.** Each section has a clear heading and explanation before the code.

| Section | What it does |
|---|---|
| 0 | Install dependencies |
| 1 | Load & explore dataset |
| 2 | Extract CLIP embeddings |
| 3 | Cluster narratives (UML phase) |
| 4 | Build social graph + GNN embeddings |
| 5 | Train classifier (SML phase) |
| 6 | Evaluate + plot results |
| 7 | Explainability — SHAP + GradCAM |
| 8 | Save model + run dashboard |


## 📦 Section 0 — Install Dependencies
Run this once. Restart kernel after if prompted.

In [ ]:
# Core ML
import subprocess, sys
pkgs = [
    "torch torchvision",
    "transformers ftfy regex tqdm",
    "git+https://github.com/openai/CLIP.git",
    "torch-geometric",
    "hdbscan umap-learn",
    "shap",
    "grad-cam",
    "fastapi uvicorn python-multipart",
    "streamlit plotly",
    "wandb",
    "pandas numpy Pillow datasets",
    "scikit-learn matplotlib seaborn",
    "pyyaml",
]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkg.split())

print("✅ All packages installed!")

## 📂 Section 1 — Load Dataset

We use **NewsCLIPpings** — a benchmark of real news articles where images and captions have been mismatched to create fake samples.

**What the data looks like:**
- `image_path` → path to the image file
- `text` → the news caption
- `label` → 0 = real, 1 = misinformation
- `post_id`, `hashtags`, `user_id` → metadata for the social graph

> ⚠️ **First time only:** Run the git clone cell below to download the dataset.


In [ ]:
import os

# ── Download dataset (run once) ──────────────────────────────────
if not os.path.exists("news_clippings"):
    os.system("git clone https://github.com/g-luo/news_clippings.git")
    os.system("cd news_clippings && pip install -q -r requirements.txt")
    os.system("cd news_clippings && python download_images.py")  # downloads images
    print("✅ Dataset downloaded!")
else:
    print("✅ Dataset already exists, skipping download.")

In [ ]:
import os
print(os.getcwd())

In [ ]:
import json
with open("news_clippings/news_clippings/data/merged_balanced/train.json") as f:
    data = json.load(f)

# Check the keys
print(data.keys())
print()
# Look at the first item
print(data["annotations"][0])

In [ ]:
import pandas as pd
import json

def load_dataset(split="train"):
    json_path = f"news_clippings/news_clippings/data/merged_balanced/{split}.json"

    with open(json_path) as f:
        data = json.load(f)

    rows = []
    for item in data["annotations"]:
        image_path = f"news_clippings/images/{item['image_id']}.jpg"

        rows.append({
            "post_id":    str(item["id"]),
            "image_path": image_path,
            "text":       "",           # no caption in this file, we'll fix later
            "label":      1 if item["falsified"] else 0,
            "hashtags":   [],
            "user_id":    "unknown"
        })

    df = pd.DataFrame(rows)
    print(f"[{split}] Loaded {len(df)} samples | "
          f"Fake: {df.label.sum()} ({df.label.mean():.1%}) | "
          f"Real: {(df.label==0).sum()}")
    return df


df_train = load_dataset("train")
df_val   = load_dataset("val")
df_test  = load_dataset("test")

# Combine for embedding extraction — we split again later
import pandas as pd
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
print(f"\nTotal samples: {len(df_all)}")
df_all.head()

In [ ]:
# ── Quick EDA ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
axes[0].bar(["Real (0)", "Fake (1)"],
            [len(df_all[df_all.label==0]), len(df_all[df_all.label==1])],
            color=["#2a9d8f", "#e63946"])
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("Count")

# Caption length distribution
df_all["caption_len"] = df_all["text"].str.split().str.len()
axes[1].hist(df_all[df_all.label==0]["caption_len"], bins=40,
             alpha=0.7, label="Real", color="#2a9d8f")
axes[1].hist(df_all[df_all.label==1]["caption_len"], bins=40,
             alpha=0.7, label="Fake", color="#e63946")
axes[1].set_title("Caption Length by Label")
axes[1].set_xlabel("Words")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150)
plt.show()
print("✅ EDA complete!")

## 🔢 Section 2 — Extract CLIP Embeddings

**What CLIP does:**
- Takes an image → converts it to a 512-dimensional number vector
- Takes a caption → converts it to another 512-dimensional number vector
- Both vectors live in the *same* space, so similar image+text pairs are close together

**What we do:**
- Run every post through CLIP
- Concatenate image + text vectors → 1024-dimensional fused embedding
- Save to disk (so you don't re-run this every time)


In [ ]:
import torch
import clip
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load CLIP
clip_model, preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()
print("✅ CLIP loaded!")

In [ ]:
class PostDataset(Dataset):
    """Simple dataset — loads image + tokenizes text."""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = preprocess(Image.open(row["image_path"]).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)  # fallback for corrupt images

        text  = clip.tokenize([row["text"]], truncate=True)[0]
        label = torch.tensor(row["label"], dtype=torch.long)
        return image, text, label


def extract_clip_embeddings(df, batch_size=64):
    """
    Runs all posts through CLIP and returns:
      image_embs  — shape (N, 512)
      text_embs   — shape (N, 512)
      fused_embs  — shape (N, 1024)  ← image + text concatenated
      labels      — shape (N,)
    """
    dataset = PostDataset(df)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=False, num_workers=2)

    image_embs, text_embs, all_labels = [], [], []

    with torch.no_grad():
        for images, texts, labels in tqdm(loader, desc="Extracting CLIP embeddings"):
            images = images.to(DEVICE)
            texts  = texts.to(DEVICE)

            img_feat = clip_model.encode_image(images)
            txt_feat = clip_model.encode_text(texts)

            # L2 normalise — makes cosine similarity = dot product
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)

            image_embs.append(img_feat.cpu().numpy())
            text_embs.append(txt_feat.cpu().numpy())
            all_labels.append(labels.numpy())

    image_embs = np.concatenate(image_embs)
    text_embs  = np.concatenate(text_embs)
    labels     = np.concatenate(all_labels)
    fused_embs = np.concatenate([image_embs, text_embs], axis=-1)

    print(f"\n✅ Embeddings extracted!")
    print(f"   image_embs : {image_embs.shape}")
    print(f"   text_embs  : {text_embs.shape}")
    print(f"   fused_embs : {fused_embs.shape}")

    return {
        "image_embs": image_embs,
        "text_embs":  text_embs,
        "fused_embs": fused_embs,
        "labels":     labels,
    }


emb_dict = extract_clip_embeddings(df_all, batch_size=64)

# Save to disk — so you don't re-run this every session
np.save("data/processed/image_embs.npy",  emb_dict["image_embs"])
np.save("data/processed/text_embs.npy",   emb_dict["text_embs"])
np.save("data/processed/fused_embs.npy",  emb_dict["fused_embs"])
np.save("data/processed/labels.npy",      emb_dict["labels"])
print("✅ Saved embeddings to data/processed/")

In [ ]:
# ── Reload saved embeddings (skip extraction next time) ──────────
# Uncomment this block on subsequent runs to skip Section 2 above

# emb_dict = {
#     "image_embs": np.load("data/processed/image_embs.npy"),
#     "text_embs":  np.load("data/processed/text_embs.npy"),
#     "fused_embs": np.load("data/processed/fused_embs.npy"),
#     "labels":     np.load("data/processed/labels.npy"),
# }
# print("✅ Loaded saved embeddings")

In [ ]:
import os
print(os.path.exists("news_clippings/images"))
print(df_all.head())

## 🗂️ Section 3 — Narrative Clustering (UML Phase)

**What this does:**
1. **UMAP** — compresses 1024-dimensional embeddings down to 50 dimensions (like squishing a 3D object into 2D, but for data)
2. **HDBSCAN** — finds natural clusters in the compressed space without you telling it how many clusters to find

**Why this matters for your project:**
- Each cluster = a distinct *misinformation narrative* (e.g. "vaccine misinformation", "election fraud", etc.)
- This is the **unsupervised learning (UML) phase** — no labels needed
- The cluster ID becomes an extra feature for the classifier later


In [ ]:
import umap
import hdbscan
from sklearn.metrics import silhouette_score, normalized_mutual_info_score

print("Step 1/2: Reducing dimensions with UMAP (this takes ~2-5 mins)...")
reducer_50d = umap.UMAP(
    n_components=50,    # compress to 50 dims first (for clustering)
    n_neighbors=15,
    metric="cosine",    # cosine works best for CLIP embeddings
    random_state=42,
    verbose=True,
)
reduced_50d = reducer_50d.fit_transform(emb_dict["fused_embs"])
print(f"✅ Reduced: {emb_dict['fused_embs'].shape} → {reduced_50d.shape}")

In [ ]:
print("Step 2/2: Clustering with HDBSCAN...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,   # minimum posts to form a cluster
    min_samples=5,
    metric="euclidean",
    prediction_data=True,  # needed to assign new points later
)
cluster_labels = clusterer.fit_predict(reduced_50d)

n_clusters  = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_ratio = (cluster_labels == -1).mean()

print(f"\n✅ Clustering done!")
print(f"   Clusters found : {n_clusters}")
print(f"   Noise points   : {noise_ratio:.1%}  (label = -1, no cluster)")

# Evaluate clustering quality
mask = cluster_labels != -1
sil  = silhouette_score(reduced_50d[mask], cluster_labels[mask])
nmi  = normalized_mutual_info_score(emb_dict["labels"], cluster_labels)
print(f"\n   Silhouette Score : {sil:.4f}  (higher = better, max 1.0)")
print(f"   NMI with labels  : {nmi:.4f}  (higher = more aligned with fake/real)")

# Save cluster labels
np.save("data/processed/cluster_labels.npy", cluster_labels)

In [ ]:
# ── Visualise clusters in 2D ─────────────────────────────────────
print("Generating 2D UMAP visualisation (takes ~2 mins)...")
reducer_2d = umap.UMAP(n_components=2, random_state=42, metric="cosine")
coords_2d  = reducer_2d.fit_transform(emb_dict["fused_embs"])

plt.figure(figsize=(14, 8))
scatter = plt.scatter(
    coords_2d[:, 0], coords_2d[:, 1],
    c=cluster_labels,
    cmap="Spectral",
    s=3, alpha=0.6,
)
plt.colorbar(scatter, label="Cluster ID  (-1 = noise)")
plt.title(f"VerifAI — Narrative Clusters  ({n_clusters} clusters found)", fontsize=14)
plt.xlabel("UMAP dim 1")
plt.ylabel("UMAP dim 2")
plt.tight_layout()
plt.savefig("results/clusters_umap.png", dpi=150)
plt.show()
print("✅ Saved → results/clusters_umap.png")

## 🕸️ Section 4 — Social Graph + GNN Embeddings

**The idea:**
- Posts that belong to the same narrative cluster are likely part of the same misinformation campaign
- We build a **graph** where posts are nodes and edges connect posts in the same cluster
- The **Graph Attention Network (GAT)** then enriches each post's embedding with information from its neighbours — capturing *propagation patterns*

**Why this is novel:**
Most papers treat posts in isolation. We model *how they relate to each other* — which is how misinformation actually spreads.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from collections import defaultdict


# ── Define the GNN ───────────────────────────────────────────────
class PropagationGNN(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=256, num_layers=3,
                 dropout=0.3, heads=4):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        # Layer 1
        self.convs.append(GATConv(input_dim, hidden_dim // heads,
                                  heads=heads, dropout=dropout))
        self.norms.append(nn.LayerNorm(hidden_dim))

        # Middle layers
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim, hidden_dim // heads,
                                      heads=heads, dropout=dropout))
            self.norms.append(nn.LayerNorm(hidden_dim))

        # Last layer — single head
        self.convs.append(GATConv(hidden_dim, hidden_dim,
                                  heads=1, dropout=dropout))
        self.norms.append(nn.LayerNorm(hidden_dim))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index)
            x = norm(x)
            x = F.elu(x)
            x = self.dropout(x)
        return x


gnn = PropagationGNN(input_dim=1024, hidden_dim=256).to(DEVICE)
print(f"✅ GNN created — {sum(p.numel() for p in gnn.parameters()):,} parameters")

In [ ]:
# ── Build the Graph ──────────────────────────────────────────────
# Strategy: connect posts that share the same cluster
# Each post is a node; edge = same narrative cluster

def build_cluster_graph(cluster_labels, max_neighbors=5):
    """
    Connects posts in the same cluster.
    max_neighbors: how many cluster-mates each post connects to
    (keeps the graph sparse and efficient)
    """
    cluster_to_posts = defaultdict(list)
    for idx, c in enumerate(cluster_labels):
        if c != -1:   # ignore noise points
            cluster_to_posts[c].append(idx)

    edges = []
    for cluster_id, post_indices in cluster_to_posts.items():
        for i in range(len(post_indices)):
            # Connect to up to max_neighbors neighbours
            neighbours = post_indices[max(0,i-max_neighbors):i] +                          post_indices[i+1:i+max_neighbors+1]
            for j in neighbours:
                edges.append([post_indices[i], j])
                edges.append([j, post_indices[i]])  # undirected

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    print(f"✅ Graph built: {len(cluster_labels)} nodes | {edge_index.size(1)} edges")
    return edge_index


edge_index = build_cluster_graph(cluster_labels, max_neighbors=5)

In [ ]:
# ── Run GNN to get propagation-aware embeddings ──────────────────
node_features = torch.tensor(
    emb_dict["fused_embs"], dtype=torch.float32
).to(DEVICE)

edge_index_gpu = edge_index.to(DEVICE)

gnn.eval()
with torch.no_grad():
    gnn_embs = gnn(node_features, edge_index_gpu)

gnn_embs = gnn_embs.cpu().numpy()
np.save("data/processed/gnn_embs.npy", gnn_embs)
print(f"✅ GNN embeddings: {gnn_embs.shape}  (saved to data/processed/gnn_embs.npy)")

## 🧠 Section 5 — Train the Classifier (SML Phase)

**What the classifier does:**
- Takes 3 inputs: CLIP fused embedding + GNN embedding + cluster one-hot vector
- Outputs a single number between 0 and 1 = probability of being misinformation
- Trained with **Focal Loss** (handles imbalanced datasets better than plain cross-entropy)

**Train/Val/Test split:**
- 70% training, 15% validation (tune on this), 15% test (final score)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset, random_split
import torch.optim as optim
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score


# ── Classifier Model ─────────────────────────────────────────────
class VerifAIClassifier(nn.Module):
    def __init__(self, clip_dim=1024, gnn_dim=256,
                 num_clusters=50, hidden_dim=128, dropout=0.3):
        super().__init__()
        input_dim = clip_dim + gnn_dim + num_clusters
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, clip_emb, gnn_emb, cluster_oh):
        x = torch.cat([clip_emb, gnn_emb, cluster_oh], dim=-1)
        logits = self.net(x)
        return logits, torch.sigmoid(logits)


# ── Focal Loss ───────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets.float(), reduction="none"
        )
        pt = torch.where(targets == 1,
                         torch.sigmoid(logits),
                         1 - torch.sigmoid(logits))
        weight = self.alpha * (1 - pt) ** self.gamma
        return (weight * bce).mean()


# ── Metrics helper ───────────────────────────────────────────────
def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    return {
        "f1":       round(f1_score(labels, preds), 4),
        "auc_roc":  round(roc_auc_score(labels, probs), 4),
        "accuracy": round(accuracy_score(labels, preds), 4),
    }

print("✅ Model classes defined!")

In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────
n_clusters = int(cluster_labels.max()) + 1

fused_t   = torch.tensor(emb_dict["fused_embs"], dtype=torch.float32)
gnn_t     = torch.tensor(gnn_embs,               dtype=torch.float32)
labels_t  = torch.tensor(emb_dict["labels"],     dtype=torch.float32)
cluster_t = torch.tensor(cluster_labels,         dtype=torch.long).clamp(min=0)

# One-hot encode cluster IDs (noise points → cluster 0)
cluster_oh = torch.zeros(len(cluster_t), n_clusters)
cluster_oh.scatter_(1, cluster_t.unsqueeze(1), 1.0)

dataset = TensorDataset(fused_t, gnn_t, cluster_oh, labels_t)

n          = len(dataset)
train_size = int(0.70 * n)
val_size   = int(0.15 * n)
test_size  = n - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)

print(f"✅ DataLoaders ready!")
print(f"   Train : {len(train_ds):,} samples")
print(f"   Val   : {len(val_ds):,} samples")
print(f"   Test  : {len(test_ds):,} samples")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────
import os, time
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

classifier = VerifAIClassifier(
    clip_dim=1024, gnn_dim=256, num_clusters=n_clusters
).to(DEVICE)

optimizer = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = FocalLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS   = 30
patience = 5
best_val_f1   = 0
no_improve    = 0
history = {"train_loss": [], "val_f1": [], "val_auc": []}

print(f"Training for {EPOCHS} epochs on {DEVICE}...\n")

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────
    classifier.train()
    total_loss = 0
    for clip_emb, gnn_emb, cluster_oh, labels in train_loader:
        clip_emb   = clip_emb.to(DEVICE)
        gnn_emb    = gnn_emb.to(DEVICE)
        cluster_oh = cluster_oh.to(DEVICE)
        labels     = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = classifier(clip_emb, gnn_emb, cluster_oh)
        loss = criterion(logits.squeeze(), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # ── Validate ─────────────────────────────────────────────────
    classifier.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for clip_emb, gnn_emb, cluster_oh, labels in val_loader:
            clip_emb   = clip_emb.to(DEVICE)
            gnn_emb    = gnn_emb.to(DEVICE)
            cluster_oh = cluster_oh.to(DEVICE)
            _, probs = classifier(clip_emb, gnn_emb, cluster_oh)
            all_probs.extend(probs.squeeze().cpu().numpy())
            all_labels.extend(labels.numpy())

    val_metrics = compute_metrics(
        np.array(all_probs), np.array(all_labels)
    )
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_f1"].append(val_metrics["f1"])
    history["val_auc"].append(val_metrics["auc_roc"])

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} | "
          f"Val F1: {val_metrics['f1']:.4f} | "
          f"Val AUC: {val_metrics['auc_roc']:.4f}")

    # ── Save best model ──────────────────────────────────────────
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        no_improve  = 0
        torch.save({
            "classifier": classifier.state_dict(),
            "gnn":        gnn.state_dict(),
            "n_clusters": n_clusters,
            "epoch":      epoch,
            "val_f1":     best_val_f1,
        }, "models/verifai_best.pt")
        print(f"   ✅ Best model saved (Val F1 = {best_val_f1:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\n⏹️  Early stopping at epoch {epoch} "
                  f"(no improvement for {patience} epochs)")
            break

print(f"\n🏁 Training complete! Best Val F1 = {best_val_f1:.4f}")

## 📊 Section 6 — Evaluate & Plot Results

Load the best saved model and run it on the held-out test set.


In [ ]:
# ── Load best model ──────────────────────────────────────────────
checkpoint = torch.load("models/verifai_best.pt", map_location=DEVICE)
classifier.load_state_dict(checkpoint["classifier"])
gnn.load_state_dict(checkpoint["gnn"])
classifier.eval()
gnn.eval()
print(f"✅ Loaded best model from epoch {checkpoint['epoch']} "
      f"(Val F1 = {checkpoint['val_f1']:.4f})")

In [ ]:
# ── Test set evaluation ──────────────────────────────────────────
all_probs, all_labels = [], []
with torch.no_grad():
    for clip_emb, gnn_emb, cluster_oh, labels in test_loader:
        clip_emb   = clip_emb.to(DEVICE)
        gnn_emb    = gnn_emb.to(DEVICE)
        cluster_oh = cluster_oh.to(DEVICE)
        _, probs = classifier(clip_emb, gnn_emb, cluster_oh)
        all_probs.extend(probs.squeeze().cpu().numpy())
        all_labels.extend(labels.numpy())

test_probs  = np.array(all_probs)
test_labels = np.array(all_labels)
test_metrics = compute_metrics(test_probs, test_labels)

print("=" * 40)
print("        TEST SET RESULTS")
print("=" * 40)
for k, v in test_metrics.items():
    print(f"  {k:<12} : {v:.4f}")
print("=" * 40)

In [ ]:
# ── Training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], color="#e63946")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Focal Loss")

axes[1].plot(history["val_f1"], color="#2a9d8f")
axes[1].set_title("Validation F1 Score")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")

axes[2].plot(history["val_auc"], color="#457b9d")
axes[2].set_title("Validation AUC-ROC")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("AUC")

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()
print("✅ Saved → results/training_curves.png")

In [ ]:
# ── Ablation study: text-only vs image-only vs full ───────────────
# This is key for your report — shows each component adds value
print("Running ablation study...")

ablation_results = {}

for name, use_img, use_txt in [
    ("Text only",        False, True),
    ("Image only",       True,  False),
    ("CLIP fused",       True,  True),
    ("+ GNN + Cluster",  True,  True),  # full VerifAI
]:
    probs_list, labels_list = [], []
    with torch.no_grad():
        for clip_emb, gnn_emb, cluster_oh, labels in test_loader:
            if not use_img:
                clip_emb[:, :512] = 0   # zero out image half
            if not use_txt:
                clip_emb[:, 512:] = 0   # zero out text half

            if name != "+ GNN + Cluster":
                gnn_emb    = torch.zeros_like(gnn_emb)
                cluster_oh = torch.zeros_like(cluster_oh)

            clip_emb   = clip_emb.to(DEVICE)
            gnn_emb    = gnn_emb.to(DEVICE)
            cluster_oh = cluster_oh.to(DEVICE)

            _, probs = classifier(clip_emb, gnn_emb, cluster_oh)
            probs_list.extend(probs.squeeze().cpu().numpy())
            labels_list.extend(labels.numpy())

    m = compute_metrics(np.array(probs_list), np.array(labels_list))
    ablation_results[name] = m
    print(f"  {name:<20} F1={m['f1']:.4f}  AUC={m['auc_roc']:.4f}")

# Bar chart
names = list(ablation_results.keys())
f1s   = [v["f1"] for v in ablation_results.values()]
aucs  = [v["auc_roc"] for v in ablation_results.values()]

x = range(len(names))
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - 0.2 for i in x], f1s,  0.4, label="F1",      color="#2a9d8f")
bars2 = ax.bar([i + 0.2 for i in x], aucs, 0.4, label="AUC-ROC", color="#457b9d")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Ablation Study — Component Contribution")
ax.legend()
plt.tight_layout()
plt.savefig("results/ablation_study.png", dpi=150)
plt.show()
print("✅ Saved → results/ablation_study.png")

## 🧠 Section 7 — Explainability (SHAP + GradCAM)

**SHAP** — answers: *which words made the model say this is fake?*
**GradCAM** — answers: *which part of the image made the model suspicious?*

These make your project stand out because you're not just predicting — you're explaining.


In [ ]:
import shap

def explain_text_shap(text, classifier, clip_model, cluster_oh_vec, device):
    """
    Returns word importance scores using SHAP.
    cluster_oh_vec: 1D tensor of shape (n_clusters,)
    """
    words = text.split()

    def predict_fn(texts):
        """Scores a list of (masked) text strings."""
        scores = []
        for t in texts:
            if not t.strip():
                t = "unknown"
            with torch.no_grad():
                tokens  = clip.tokenize([t], truncate=True).to(device)
                txt_emb = clip_model.encode_text(tokens)
                txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
                img_emb = torch.zeros_like(txt_emb)
                fused   = torch.cat([img_emb, txt_emb], dim=-1)
                gnn_emb = torch.zeros(1, 256).to(device)
                _, prob = classifier(fused, gnn_emb,
                                     cluster_oh_vec.unsqueeze(0).to(device))
            scores.append(prob.item())
        return np.array(scores)

    masker    = shap.maskers.Text(tokenizer=lambda x: x.split())
    explainer = shap.Explainer(predict_fn, masker)
    shap_vals = explainer([text])

    word_importance = dict(zip(
        shap_vals.data[0],
        shap_vals.values[0].tolist()
    ))
    return word_importance, shap_vals


# ── Test on a sample post ────────────────────────────────────────
sample_idx  = 0
sample_row  = df_all.iloc[sample_idx]
sample_oh   = cluster_oh[sample_idx]

print(f"Caption : {sample_row['text']}")
print(f"Label   : {'FAKE' if sample_row['label'] == 1 else 'REAL'}")

word_imp, shap_vals = explain_text_shap(
    sample_row["text"], classifier, clip_model, sample_oh, DEVICE
)

# Plot SHAP waterfall
shap.plots.text(shap_vals)
print("\nTop words pushing toward FAKE:")
sorted_words = sorted(word_imp.items(), key=lambda x: x[1], reverse=True)
for word, score in sorted_words[:5]:
    direction = "→ FAKE" if score > 0 else "→ REAL"
    print(f"  '{word}': {score:+.4f}  {direction}")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import transforms as T

def explain_image_gradcam(pil_image, clip_model, classifier,
                           cluster_oh_vec, device):
    """
    Generates a GradCAM heatmap highlighting suspicious image regions.
    Returns: numpy array (224, 224, 3) — overlay of heatmap on image
    """
    preproc = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.48145466, 0.4578275,  0.40821073],
                    std= [0.26862954, 0.26130258, 0.27577711]),
    ])

    img_tensor = preproc(pil_image).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Target = last transformer block in CLIP vision encoder
    target_layer = clip_model.visual.transformer.resblocks[-1]

    class CLIPWrapper(nn.Module):
        """Wraps CLIP + classifier for GradCAM compatibility."""
        def __init__(self):
            super().__init__()
        def forward(self, x):
            img_emb = clip_model.encode_image(x).float()
            img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
            txt_emb = torch.zeros_like(img_emb)
            fused   = torch.cat([img_emb, txt_emb], dim=-1)
            gnn_emb = torch.zeros(x.size(0), 256).to(device)
            logits, _ = classifier(fused, gnn_emb,
                                    cluster_oh_vec.unsqueeze(0).to(device))
            return logits

    wrapper = CLIPWrapper().to(device)
    cam     = GradCAM(model=wrapper, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=img_tensor)[0]

    img_np = np.array(pil_image.resize((224, 224))).astype(np.float32) / 255.0
    overlay = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    return overlay, grayscale_cam


# ── Test on a sample post ────────────────────────────────────────
sample_image = Image.open(sample_row["image_path"]).convert("RGB")
overlay, _   = explain_image_gradcam(
    sample_image, clip_model, classifier, sample_oh, DEVICE
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample_image.resize((224, 224)))
axes[0].set_title("Original Image")
axes[0].axis("off")
axes[1].imshow(overlay)
axes[1].set_title("GradCAM — Suspicious Regions (red = high attention)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("results/gradcam_example.png", dpi=150)
plt.show()
print("✅ Saved → results/gradcam_example.png")

## 💾 Section 8 — Save Model & Run Dashboard

Once training is done, save everything and launch the Streamlit dashboard.


In [ ]:
# ── Save full model package ──────────────────────────────────────
import pickle

save_pkg = {
    "classifier":     classifier.state_dict(),
    "gnn":            gnn.state_dict(),
    "n_clusters":     n_clusters,
    "reducer_50d":    reducer_50d,   # UMAP reducer for new posts
    "clusterer":      clusterer,     # HDBSCAN for new posts
    "clip_model_name": "ViT-B/32",
    "best_val_f1":    checkpoint["val_f1"],
    "test_metrics":   test_metrics,
}
torch.save(save_pkg, "models/verifai_full.pt")

# Also save UMAP/HDBSCAN separately (they're sklearn objects)
with open("models/umap_reducer.pkl", "wb") as f:
    pickle.dump(reducer_50d, f)
with open("models/hdbscan_clusterer.pkl", "wb") as f:
    pickle.dump(clusterer, f)

print("✅ Full model package saved to models/")
print("\nFinal Results Summary:")
print(f"  Test F1      : {test_metrics['f1']:.4f}")
print(f"  Test AUC-ROC : {test_metrics['auc_roc']:.4f}")
print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")

In [ ]:
# ── Launch Streamlit Dashboard ───────────────────────────────────
# Run this in your terminal (not inside Jupyter):
#
#   streamlit run dashboard/app.py
#
# The dashboard will open at http://localhost:8501

print("To launch the dashboard, run in your terminal:")
print()
print("   streamlit run dashboard/app.py")
print()
print("To launch the API, run:")
print()
print("   uvicorn src.api.main:app --reload --port 8000")
print("   Then open: http://localhost:8000/docs")

## 🎉 You're Done!

Here's what you've built:

| Component | Status |
|---|---|
| CLIP multimodal embeddings | ✅ |
| HDBSCAN narrative clustering (UML) | ✅ |
| Graph Attention Network | ✅ |
| Misinformation classifier (SML) | ✅ |
| Focal Loss + early stopping | ✅ |
| Ablation study | ✅ |
| SHAP text explainability | ✅ |
| GradCAM image explainability | ✅ |
| Saved model checkpoint | ✅ |

**Your resume line:**
> *Built VerifAI, an end-to-end multimodal misinformation detection system combining CLIP embeddings, Graph Attention Networks, and XAI (SHAP + GradCAM). Achieved [your F1] F1 on NewsCLIPpings benchmark. Deployed as REST API + Streamlit dashboard.*
